In [1]:
import numpy as np
import tensorflow as tf

In [2]:
features_train_conjunctiva = np.load("dense_features_train.npy")
#features_train_palm = np.load("dense_features_train_palm.npy")
#features_train_nail = np.load("dense_features_train_nail.npy")


In [3]:
conjunctiva_train = tf.convert_to_tensor(features_train_conjunctiva, dtype=tf.float32)
#palm_train = tf.convert_to_tensor(features_train_palm, dtype=tf.float32)
#nail_train = tf.convert_to_tensor(features_train_nail, dtype=tf.float32)


# Self Attention

In [4]:
def self_attention_block(x, d_model=1024):
    Q = tf.keras.layers.Dense(d_model)(x)
    K = tf.keras.layers.Dense(d_model)(x)
    V = tf.keras.layers.Dense(d_model)(x)
    
    scores = tf.matmul(Q, K, transpose_b=True) / tf.math.sqrt(tf.cast(d_model, tf.float32))
    weights = tf.nn.softmax(scores, axis=-1)
    out = tf.matmul(weights, V)
    
    out = tf.keras.layers.LayerNormalization()(x + out)
    return out

In [5]:
conj_self = self_attention_block(conjunctiva_train)
#palm_self = self_attention_block(palm_train)
#nail_self = self_attention_block(nail_train)

# Cross Attention

In [ ]:
def cross_attention_block(query, key_value, d_model=1024):
    Q = tf.keras.layers.Dense(d_model)(query)
    K = tf.keras.layers.Dense(d_model)(key_value)
    V = tf.keras.layers.Dense(d_model)(key_value)
    
    scores = tf.matmul(Q, K, transpose_b=True) / tf.math.sqrt(tf.cast(d_model, tf.float32))
    weights = tf.nn.softmax(scores, axis=-1)
    out = tf.matmul(weights, V)
    out = tf.keras.layers.LayerNormalization()(query + out)
    return out

In [ ]:
text_cross_conj = cross_attention_block(text_self, conj_self)
text_cross_palm = cross_attention_block(text_self, palm_self)
text_cross_nail = cross_attention_block(text_self, nail_self)

In [ ]:
fusion_vector = tf.concat([conj_self, palm_self, nail_self, text_emb], axis=-1)

x = Dense(512, activation='relu')(fusion_vector)
x = tf.keras.layers.Dropout(0.5)(x)

output_class = Dense(1, activation='sigmoid')(x)  # anemia classification
output_hb = Dense(1, activation='linear')(x)      # hemoglobin regression

final_model = tf.keras.Model(
    inputs=[conj_input, palm_input, nail_input, text_input],
    outputs=[output_class, output_hb]
)